In [0]:
# O QUE: Definir as fontes da camada Bronze.
# POR QUÊ: Centralizar a configuração dos arquivos Olist.

caminho_raw = "/Volumes/workspace/default/olist_raw/"

fontes = {
    "olist_customers_dataset.csv": "customers",
    "olist_geolocation_dataset.csv": "geolocation",
    "olist_order_items_dataset.csv": "order_items",
    "olist_order_payments_dataset.csv": "order_payments",
    "olist_order_reviews_dataset.csv": "order_reviews",
    "olist_products_dataset.csv": "products",
    "olist_sellers_dataset.csv": "sellers",
    "product_category_name_translation.csv": "product_category_name_translation"
}

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")


In [0]:

# O QUE: Criar as tabelas Bronze que ainda não existem.
# POR QUÊ: Armazenar os dados originais em Delta Lake
# e evitar sobrescritas acidentais.

for arquivo_csv, nome_tabela in fontes.items():

    tabela = f"workspace.bronze.{nome_tabela}"

    if spark.catalog.tableExists(tabela):
        print(f"JÁ EXISTE: {tabela}")
        continue

    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("multiLine", "true")
        .option("quote", '"')
        .option("escape", '"')
        .option("inferSchema", "false")
        .load(f"{caminho_raw}{arquivo_csv}")
    )

    df.write.format("delta").saveAsTable(tabela)

    print(f"CRIADA: {tabela} | Registros: {spark.table(tabela).count()}")

print("\nTabelas Bronze disponíveis:")
display(spark.sql("SHOW TABLES IN workspace.bronze"))


In [0]:

# O QUE: Validar as nove tabelas Bronze.
# POR QUÊ: Confirmar registros e formato Delta.

resultados = []

for nome in ["orders"] + list(fontes.values()):
    tabela = f"workspace.bronze.{nome}"

    total = spark.table(tabela).count()

    detalhe = spark.sql(
        f"DESCRIBE DETAIL {tabela}"
    ).select("format").first()

    resultados.append((nome, total, detalhe["format"]))

df_validacao = spark.createDataFrame(
    resultados,
    ["tabela", "registros", "formato"]
)

display(df_validacao.orderBy("tabela"))
